# Load data on MRIs from ASCE 7 hazard tool

In [24]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import weibull_min
from openpyxl import load_workbook

# Square root function for 
def sqrt_func(x, a):
        return a * np.sqrt(x) + 1

# Define the linearized Weibull function
def linear_weibull(log_x, k, log_lambda):
    return k * log_x - k * log_lambda
    
def compute_footprint_factors():

    ''' Get factors for adjusting building footprints
    This function will load your return period data from the ASCE 7 hazard tool and calculate the building footprint factors
    These will then be written into the community data file'''

    # ================================
    # =========== Load and format data
    # ================================

    '''The data file contains the wind speeds for different MRIs and footprint sizes. Units are in ft2 for footrpint 
    and mph for wind speed'''
    
    data = pd.read_excel('return_periods.xlsx') # Update this file for a new area of interest

    # Get effective area, Mean recurrance interval, and wind speeds
    Ae = np.array(data.iloc[1:,0:1]).flatten()
    MRI = np.array(data.iloc[0:1,1:]).flatten()
    winds = np.array(data.iloc[1:,1:])

    Ae_m2 = Ae*0.092903 # convert effective area from ft2 to m2
    winds_ms = winds*0.447 # converts winds from mph to m/s
    non_exceedance_prob = 1-1/MRI # converts MRI to exceedance probability

    # ====================================
    # =========== Fit Weibull distribution
    # ====================================

    # Define for all footprint sizes (8 total)
    k_fitted = np.zeros(8)
    lambda_fitted = np.zeros(8)
    cdf_fit = np.zeros([125,8])

    # Loop through all footprint size
    for i in np.arange(8):
        x = winds_ms[i,:][~np.isnan(winds_ms[i,:])]
        exceedance_prob = 1 - non_exceedance_prob[~np.isnan(winds_ms[i,:])] # Get exceedance probability
        
        # Transform to log-log scale
        log_x = np.log(x)
        log_negative_log_S = np.log(-np.log(exceedance_prob))
        
        # Fit the linearized model
        params, _ = curve_fit(linear_weibull, log_x, log_negative_log_S)

        # get lambda and k parameters
        k_fitted[i], log_lambda_fitted = params
        lambda_fitted[i] = np.exp(log_lambda_fitted)
    
        # Generate fitted CDF
        x_fit = np.linspace(0, 125,125) # Get CDF for range of wind speeds
        
        cdf_fit[:,i] = 1 - np.exp(-(x_fit / lambda_fitted[i])**k_fitted[i])

    # =========================================
    # =============== Calculate scaling factors
    # =========================================
    
    ef_vals = np.array([29,38,50,61,75,89]) #EF0 to 5
    factors = np.zeros([8,6])
    
    for i in np.arange(6):
        factors[:,i] = (1-cdf_fit[ef_vals[i],:])/(1-cdf_fit[ef_vals[i],0])

    # =====================================
    # ============ Fit square root function
    # =====================================
    
    # Initial guess for the parameters [a, b, c]
    p0 = [1]

    params = np.zeros(6)
    for i in np.arange(6):
        params[i], covariance = curve_fit(sqrt_func, Ae_m2, factors[:,i], p0=p0)

    
    # ==================================================
    # ============= Interpolate factor for each archetype
    # ==================================================

    # Load building footprint size
    building_data = pd.read_excel("community_data.xlsx",sheet_name='FloorAreas')
    building_footprints = building_data['FA']

    # Get factors
    
    factors_building = np.zeros([params.shape[0],building_footprints.shape[0]])
    for i in np.arange(6):
        factors_building[i,:] = params[i]*np.sqrt(building_footprints)+1

    # Average across wind intensities
    factors_building = factors_building.mean(axis=0)

    # ==============================
    # =============== Write results
    # ==============================

    
    df = pd.DataFrame(factors_building)
    df.columns=['Factor']
    df.index = ['A1','A5','A6','A7','A8','A9','A10','A11','A12','A13','A14','A15','A16','A17','A18','A19']

    with pd.ExcelWriter('community_data.xlsx', mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name="Footprint_factors")

    